# this notebook is for merging defensive stats with the actual data

In [121]:
# necessary imports
import pandas as pd
import numpy as np
import difflib
import re

In [122]:
# loading both datasets
df_def = pd.read_csv("defensive_stats_cleaned.csv")
df_main = pd.read_csv("all_seasons_data.csv")

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_13608\3027888008.py:3: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df_main = pd.read_csv("all_seasons_data.csv")


In [ ]:
key_cols = ["name", "season", "GW"]
# identify duplicated rows based on key columns
dup_mask = df_main.duplicated(subset=key_cols, keep=False)

print("Number of duplicated rows:", dup_mask.sum())
duplicated_rows = df_main[dup_mask].sort_values(by=key_cols)
# print sample data to verify with the name, season, and GW minutes
target_cols = key_cols + ["minutes",'fixture']
print(duplicated_rows[target_cols].head(10))

Number of duplicated rows: 16856
                  name   season  GW  minutes  fixture
127104  Aaron Connolly  2021-22  22        0      212
127105  Aaron Connolly  2021-22  22        0      232
129627  Aaron Connolly  2021-22  25        0      250
129628  Aaron Connolly  2021-22  25        0      174
133180  Aaron Connolly  2021-22  29        0      283
133181  Aaron Connolly  2021-22  29        0      153
135394  Aaron Connolly  2021-22  33        0      327
135395  Aaron Connolly  2021-22  33        0      295
194935  Aaron Connolly  2023-24  37        0      366
194936  Aaron Connolly  2023-24  37        0      332


In [131]:
key_cols = ["name", "season", "GW"]

dup_mask = df_def.duplicated(subset=key_cols, keep=False)

print("Number of duplicated rows:", dup_mask.sum())
duplicated_rows = df_def[dup_mask].sort_values(by=key_cols)
# print sample data to verify with the name, season, and GW minutes
target_cols = key_cols + ["minutes",'game']
print(duplicated_rows[target_cols].head(10))

Number of duplicated rows: 25428
                name   season  GW  minutes  \
5143  Aaron Connolly  2019-20  22     67.0   
5503  Aaron Connolly  2019-20  22     45.0   
6059  Aaron Connolly  2019-20  25     67.0   
6328  Aaron Connolly  2019-20  25     31.0   
8065  Aaron Connolly  2019-20  38     20.0   
8259  Aaron Connolly  2019-20  38     81.0   
8638  Aaron Connolly  2019-20  38     86.0   
8943  Aaron Connolly  2019-20  38     71.0   
9233  Aaron Connolly  2019-20  38     20.0   
9433  Aaron Connolly  2019-20  38     58.0   

                                     game  
5143        2019-12-26 Tottenham-Brighton  
5503          2020-01-01 Brighton-Chelsea  
6059      2020-01-18 Brighton-Aston Villa  
6328      2020-01-21 Bournemouth-Brighton  
8065          2020-06-20 Brighton-Arsenal  
8259   2020-06-23 Leicester City-Brighton  
8638   2020-06-30 Brighton-Manchester Utd  
8943     2020-07-04 Norwich City-Brighton  
9233        2020-07-08 Brighton-Liverpool  
9433  2020-07-11 Bri

In [102]:

# ==========================================
# STEP 1: FORCE-CLEAN NAMES
# ==========================================

def clean_main_name_format(name):
    if not isinstance(name, str):
        return str(name)
    
    # 1. Fix Mojibake (Encoding Errors)
    char_map = {
        'Ã©': 'é', 'Ãº': 'ú', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã¨': 'è', 'Ã±': 'ñ',
        'Ã\xad': 'í', 'Ã§': 'ç', 'Ã¢': 'â', 'Ã¼': 'ü', 'Ã¶': 'ö', 'Ã\x9f': 'ß',
        'Ã¸': 'ø', 'Ã«': 'ë', 'Ã': 'à' ,'à£': 'ã', 'à©': 'é'
    }
    for bad, good in char_map.items():
        name = name.replace(bad, good)

    # 2. Remove trailing IDs (e.g., '_376', '_12', '_4')
    # Regex: Underscore followed by 1 or more digits at the END of the string
    name = re.sub(r'_\d+$', '', name)
    
    # 3. Replace remaining underscores with spaces
    name = name.replace('_', ' ')
    
    # 4. Standardize (lower, strip)
    return name.lower().strip()

print("Applying name cleaning...")
# Create/Overwrite the 'join_name' column
df_main['join_name'] = df_main['name'].apply(clean_main_name_format)


# ==========================================
# VERIFICATION & VALIDATION
# ==========================================
print("\n=== VALIDATION REPORT ===")

# Check 1: Are there any underscores left? (Should be 0)
underscores_left = df_main[df_main['join_name'].str.contains('_')].shape[0]
print(f"Rows with underscores ('_') remaining: {underscores_left}")

# Check 2: Are there any digits left? (Should be 0, unless a name legitimately has numbers)
digits_left = df_main[df_main['join_name'].str.contains(r'\d')].shape[0]
print(f"Rows with digits remaining: {digits_left}")

# Check 3: Specific Examples (The ones that failed before)
targets = ['Cresswell', 'Doucour', 'Lennon']
print("\nSpecific Check (Before vs After):")
for t in targets:
    # Find a row containing this text in the ORIGINAL name
    sample = df_main[df_main['name'].str.contains(t, na=False)].head(1)
    if not sample.empty:
        orig = sample['name'].values[0]
        clean = sample['join_name'].values[0]
        print(f"Original: '{orig}'  ->  Cleaned: '{clean}'")

# Check 4: Encoding Check
# Search for the bad character 'Ã'
bad_char_count = df_main[df_main['join_name'].str.contains('Ã', na=False)].shape[0]
print(f"\nRows with broken encoding ('Ã') remaining: {bad_char_count}")

if underscores_left == 0 and digits_left == 0 and bad_char_count == 0:
    print("\n✅ SUCCESS: Name format is clean.")
else:
    print("\n❌ WARNING: Some names are still dirty. Check the counters above.")

Applying name cleaning...

=== VALIDATION REPORT ===
Rows with underscores ('_') remaining: 0
Rows with digits remaining: 0

Specific Check (Before vs After):
Original: 'Aaron_Cresswell'  ->  Cleaned: 'aaron cresswell'
Original: 'Abdoulaye_Doucouré'  ->  Cleaned: 'abdoulaye doucouré'
Original: 'Aaron_Lennon'  ->  Cleaned: 'aaron lennon'

Rows with broken encoding ('Ã') remaining: 0

✅ SUCCESS: Name format is clean.


In [103]:
# first filter only the seasons that we need from defensive data
seasons_needed = df_def['season'].unique()
df_main = df_main[df_main['season'].isin(seasons_needed)]
# print the target seasons
print("Seasons needed for merging:", seasons_needed)

Seasons needed for merging: ['2019-20' '2020-21' '2021-22' '2022-23' '2023-24' '2024-25']


In [104]:
# second compare the columns and datatypes
print("Defensive DataFrame columns and dtypes:")
print(df_def.dtypes)
print("\nMain DataFrame columns and dtypes:")
print(df_main.dtypes)

Defensive DataFrame columns and dtypes:
season                      object
name                        object
team                        object
position                    object
minutes                    float64
tackles                      int64
tackles_won                  int64
tackles_total                int64
challenges                   int64
challenges_attempted         int64
challenges_success_rate    float64
challenges_lost              int64
blocks                       int64
blocks_shots                 int64
blocks_passes                int64
interceptions              float64
tackles_interceptions      float64
clearances                 float64
errors                     float64
match_id                    object
game                        object
game_date                   object
GW                           int64
dtype: object

Main DataFrame columns and dtypes:
Unnamed: 0                           int64
name                                object
assists            

In [105]:
#clearances_blocks_interceptions, recoveries, defensive_contribution, tackles are the cols needed to be added to the defensive df

def add_clearances_blocks_interceptions(df):
    df['clearances_blocks_interceptions'] = df['clearances'] + df['blocks'] + df['interceptions']
    return df
df_def = add_clearances_blocks_interceptions(df_def)
# print sample data to verify
print(df_def[['clearances', 'blocks', 'interceptions', 'clearances_blocks_interceptions']].head())

   clearances  blocks  interceptions  clearances_blocks_interceptions
0         0.0       0            0.0                              0.0
1         0.0       0            0.0                              0.0
2         1.0       2            1.0                              4.0
3         0.0       0            1.0                              1.0
4         0.0       2            2.0                              4.0


In [106]:
ghost_mask = (df_main['minutes'] == 0) & (
    (df_main['tackles'] > 0) | 
    (df_main['clearances_blocks_interceptions'] > 0)
)

# 2. Grab the subset
ghost_rows = df_main[ghost_mask]
# print sample data to verify
print(ghost_rows.head())

Empty DataFrame
Columns: [Unnamed: 0, name, assists, bonus, bps, clean_sheets, clearances_blocks_interceptions, creativity, element, fixture, goals_conceded, goals_scored, ict_index, influence, kickoff_time, minutes, opponent_team, own_goals, penalties_missed, penalties_saved, recoveries, red_cards, round, saves, selected, tackles, team_a_score, team_h_score, threat, total_points, transfers_balance, transfers_in, transfers_out, value, was_home, yellow_cards, GW, position, team, defensive_contribution, season, join_name]
Index: []

[0 rows x 42 columns]


In [107]:
# ==========================================
# STEP 2: STANDARDIZE KEYS & APPLY MAPPING
# ==========================================
print("--- STANDARDIZING KEYS ---")

# 1. Standardize Defensive Names (Simple lower/strip)
# (Main DF is already cleaned from Step 1)
df_def['join_name'] = df_def['name'].astype(str).str.lower().str.strip()

# 2. Standardize Seasons (Ensure they match "2023-24" format in both)
df_main['join_season'] = df_main['season'].astype(str).str.strip()
df_def['join_season'] = df_def['season'].astype(str).str.strip()

# 3. Apply Manual Nickname Map (The "Hard Cases")
# We map the Main (Legal) names to the Defensive (Common) names for the JOIN KEY only.
# Your original 'name' column remains the formal one.
manual_nickname_map = {
    'jorge luiz frello filho': 'jorginho',
    'jonathan castro otto': 'jonny',
    'bruno guimaraes rodriguez moura': 'bruno guimaraes',
    'gabriel teodoro martinelli silva': 'gabriel martinelli',
    'emerson leie de souza junior': 'emerson royal',
    'david raya martin': 'david raya',
    'jose sa': 'jose sa',
    'joao palhinha goncalves alves': 'joao palhinha',
    'thiago alcantara do nascimento': 'thiago',
    'matheus luiz nunes': 'matheus nunes',
    'antony matheus dos santos': 'antony',
    'richarlison de andrade': 'richarlison',
    'bernardo mota veiga de carvalho e silva': 'bernardo silva',
    'ederson santana de moraes': 'ederson'
}

print(f"Applying manual fixes for {len(manual_nickname_map)} specific players...")
df_main['join_name'] = df_main['join_name'].replace(manual_nickname_map)

# ==========================================
# VERIFICATION (OVERLAP CHECK)
# ==========================================
print("\n=== MATCHING VALIDATION ===")

# Create the sets of unique names available
main_names = set(df_main['join_name'].unique())
def_names = set(df_def['join_name'].unique())

# Find intersection (Names present in BOTH)
common_names = main_names.intersection(def_names)
missing_names = main_names - def_names

# --- KEY METRICS ---
unique_def_names = len(def_names)
matched_def_names = len(common_names)

print(f"Unique Names in Main:      {len(main_names)}")
print(f"Unique Names in Defensive: {unique_def_names}")
print(f"Names Matched:             {matched_def_names}")

# 1. NAME MATCHING PERCENTAGE (Requested Metric)
# How many of the defensive players did we find in the main list?
name_match_pct = (matched_def_names / unique_def_names) * 100
print(f"Defensive Name Coverage:   {name_match_pct:.2f}% (Names in Def found in Main)")

# 2. ROW MATCHING PERCENTAGE
# Calculate coverage based on ROWS (Total data available)
main_keys = set(zip(df_main['join_name'], df_main['GW'], df_main['join_season']))
def_keys = set(zip(df_def['join_name'], df_def['GW'], df_def['join_season']))

matched_rows = len(main_keys.intersection(def_keys))
total_rows = len(df_main)

print(f"\nTotal Rows in Main:        {total_rows}")
print(f"Rows with Defensive Data:  {matched_rows}")
print(f"Row Match Percentage:      {(matched_rows / total_rows) * 100:.2f}%")

if len(missing_names) > 0:
    print("\nTop 5 Names Still Missing (Check if these are active players):")
    # We count how often the missing names appear to prioritize big misses
    missing_counts = df_main[df_main['join_name'].isin(missing_names)]['join_name'].value_counts()
    print(missing_counts.head(5))

--- STANDARDIZING KEYS ---
Applying manual fixes for 14 specific players...

=== MATCHING VALIDATION ===
Unique Names in Main:      1929
Unique Names in Defensive: 1342
Names Matched:             1067
Defensive Name Coverage:   79.51% (Names in Def found in Main)

Total Rows in Main:        142780
Rows with Defensive Data:  32704
Row Match Percentage:      22.91%

Top 5 Names Still Missing (Check if these are active players):
join_name
gabriel fernando de jesus              212
fabian schà¤r                          211
ezri konsa ngoyo                       211
joelinton cássio apolinário de lira    211
pedro lomba neto                       211
Name: count, dtype: int64


In [108]:
# ==========================================
# STEP 3: AUTOMATED SMART MATCHING
# ==========================================
print("--- STARTING SMART MATCHING ---")

# 1. PREPARE LISTS
# We only care about names that are currently missing in the Main DF
# (i.e., names in Main that don't yet match a name in Defensive)
valid_def_names = set(df_def['join_name'].unique())
main_unique = df_main['join_name'].unique()
missing_names = [n for n in main_unique if n not in valid_def_names]

print(f"Attempting to resolve {len(missing_names)} missing names...")

name_mapping = {}

# 2. LOGIC A: SUBSTRING MATCH (The "Gabriel Jesus" Fix)
# We check if a Defensive Name (Short) is fully inside a Main Name (Long)
# e.g. "gabriel jesus" is inside "gabriel fernando de jesus"
for m_name in missing_names:
    m_tokens = set(m_name.split())
    candidates = []
    
    for d_name in valid_def_names:
        d_tokens = set(d_name.split())
        # Check if ALL words in the short name appear in the long name
        if d_tokens.issubset(m_tokens):
            candidates.append(d_name)
    
    if candidates:
        # If multiple matches, pick the longest one (Most specific)
        # Prevents "Gabriel" matching "Gabriel Jesus" incorrectly
        best_match = max(candidates, key=len)
        name_mapping[m_name] = best_match

# 3. LOGIC B: FUZZY MATCH (The "Fabian Schär" Fix)
# For names that didn't match via substring (likely due to spelling/encoding diffs)
# We only check names that Logic A didn't solve
remaining_missing = [n for n in missing_names if n not in name_mapping]
def_name_list = list(valid_def_names)

for m_name in remaining_missing:
    # Cutoff 0.8 is strict to avoid bad matches (we prefer missing data over wrong data)
    matches = difflib.get_close_matches(m_name, def_name_list, n=1, cutoff=0.8)
    if matches:
        name_mapping[m_name] = matches[0]

# 4. APPLY THE UPDATES (To the JOIN KEY only)
print(f"Found {len(name_mapping)} new automatic matches.")
print("Updating 'join_name' column (Original names are safe)...")
df_main['join_name'] = df_main['join_name'].replace(name_mapping)

# ==========================================
# FINAL VERIFICATION
# ==========================================
print("\n=== FINAL MATCH VALIDATION ===")

# Recalculate metrics
main_names = set(df_main['join_name'].unique())
def_names = set(df_def['join_name'].unique())
common_names = main_names.intersection(def_names)

# Coverage Metrics
unique_def_names = len(def_names)
matched_def_names = len(common_names)
name_match_pct = (matched_def_names / unique_def_names) * 100

print(f"Names Matched:             {matched_def_names}")
print(f"Defensive Name Coverage:   {name_match_pct:.2f}% (Names in Def found in Main)")

# Row Metrics
main_keys = set(zip(df_main['join_name'], df_main['GW'], df_main['join_season']))
def_keys = set(zip(df_def['join_name'], df_def['GW'], df_def['join_season']))
matched_rows = len(main_keys.intersection(def_keys))
total_rows = len(df_main)

print(f"Row Match Percentage:      {(matched_rows / total_rows) * 100:.2f}%")

# Check our "Problem Children" from the previous step
check_list = [
    'gabriel fernando de jesus', 
    'joelinton cássio apolinário de lira', 
    'fabian schà¤r'
]
print("\n--- Specific Fix Check ---")
for name in check_list:
    # We have to check if the *cleaned* version of this name was mapped
    # (Since our loop worked on 'join_name', we need to recreate the clean key to check the dict)
    clean_key = name.lower() # Simplified cleaning for this check
    
    # Check if this name is now in our dataframe as a "Defensive Name"
    # We filter the Main DF for the ORIGINAL name to see what its join_key became
    # Note: df_main['name'] is likely lowercase/cleaned by Step 1, so we search flexibly
    sample = df_main[df_main['join_name'].str.contains(clean_key.split()[0], na=False)].head(1)
    
    if name in name_mapping:
         print(f"'{name}'  ->  MAPPED TO: '{name_mapping[name]}'")
    else:
         print(f"'{name}'  ->  NOT MAPPED (Status Unknown)")

--- STARTING SMART MATCHING ---
Attempting to resolve 862 missing names...
Found 321 new automatic matches.
Updating 'join_name' column (Original names are safe)...

=== FINAL MATCH VALIDATION ===
Names Matched:             1266
Defensive Name Coverage:   94.34% (Names in Def found in Main)
Row Match Percentage:      28.08%

--- Specific Fix Check ---
'gabriel fernando de jesus'  ->  MAPPED TO: 'gabriel jesus'
'joelinton cássio apolinário de lira'  ->  MAPPED TO: 'joelinton'
'fabian schà¤r'  ->  MAPPED TO: 'fabian schär'


In [109]:
# ==========================================
# 1. DEFENSIVE LEFTOVERS (Who did we miss?)
# ==========================================
# These are names in the Defensive file that never found a partner in Main
main_names_final = set(df_main['join_name'].unique())
def_names_final = set(df_def['join_name'].unique())

unused_def_names = def_names_final - main_names_final

print("=== DEFENSIVE NAMES NOT MATCHED ===")
print(f"Total Unused Defensive Names: {len(unused_def_names)} (The remaining ~6%)")

if len(unused_def_names) > 0:
    print("\nSample of Unused Names (Likely obscure players):")
    # Sort them to see if any look familiar
    print(sorted(list(unused_def_names))[:15])

# ==========================================
# 2. MAIN DATAFRAME GAPS (Are they important?)
# ==========================================
# We verify if the unmatched rows in Main are mostly just 0-minute players

# Filter Main DF to rows that DO NOT have a match
unmatched_rows = df_main[~df_main['join_name'].isin(def_names_final)]

print("\n=== MAIN ROWS WITHOUT MATCH ===")
print(f"Total Unmatched Rows: {len(unmatched_rows)}")

# A. The "Bench" Check
bench_unmatched = len(unmatched_rows[unmatched_rows['minutes'] == 0])
active_unmatched = len(unmatched_rows[unmatched_rows['minutes'] > 0])

print(f" - Bench Rows (0 Minutes): {bench_unmatched} (Safe to ignore)")
print(f" - Active Rows (>0 Minutes): {active_unmatched} (Potential data loss)")

# B. Inspect the Active Misses
if active_unmatched > 0:
    print("\nTop 10 ACTIVE Players Missing Data (Minutes > 0):")
    # Group by name and sum minutes to see who the biggest missing active players are
    active_missing_stats = unmatched_rows[unmatched_rows['minutes'] > 0].groupby('join_name')['minutes'].sum().sort_values(ascending=False)
    print(active_missing_stats.head(10))

=== DEFENSIVE NAMES NOT MATCHED ===
Total Unused Defensive Names: 76 (The remaining ~6%)

Sample of Unused Names (Likely obscure players):
['abdukodir khusanov', 'abdul fatawu issahaku', 'ahmed hegazi', 'albert grønbaek', 'alex palmer', 'andrés garcía', 'angeliño', 'ansu fati', 'ben gannon-doak', 'beto', 'borja bastón', 'cafú', 'casemiro', 'chidozie obi-martin', 'chiquinho']

=== MAIN ROWS WITHOUT MATCH ===
Total Unmatched Rows: 19915
 - Bench Rows (0 Minutes): 17875 (Safe to ignore)
 - Active Rows (>0 Minutes): 2040 (Potential data loss)

Top 10 ACTIVE Players Missing Data (Minutes > 0):
join_name
tomas soucek                           10331
fabio henrique tavares                  9616
frederico rodrigues de paula santos     7800
benjamin white                          6735
lucas tolentino coelho de lima          6242
jonny                                   5664
raphael dias belloli                    5274
carlos henrique casimiro                4936
solomon march                     

In [110]:

# ==========================================
# 2. MAP THE MISSING STARS (Manual Dictionary)
# ==========================================
print("--- MAPPING MISSING STARS ---")

# These are the specific players from your "Top 10" list
star_player_map = {
    'joão filipe iria santos moutinho': 'joão moutinho',
    'fabio henrique tavares': 'fabinho',
    'frederico rodrigues de paula santos': 'fred',
    'lucas tolentino coelho de lima': 'lucas paquetá', # Check accent
    'lucas tolentino coelho de lima': 'lucas paqueta', # Try both if unsure
    'benjamin white': 'ben white',
    'gabriel dos santos magalhães': 'gabriel magalhães', # Now that encoding is fixed, map to short
    'bruno guimarães rodriguez moura': 'bruno guimarães',
    'joão palhinha gonçalves': 'joão palhinha',
    'tomas soucek': 'tomáš souček', # Add accents if defensive has them
    'casemiro': 'casemiro', # If Main has long name, map it. Likely 'carlos henrique casimiro'
    'carlos henrique casimiro': 'casemiro'
}

# Apply the map
df_main['join_name'] = df_main['join_name'].replace(star_player_map)


# ==========================================
# 3. VERIFY THE RESULT
# ==========================================
print("\n=== FINAL STAR CHECK ===")

# Check if these specific players are now matched
targets = ['joão moutinho', 'fabinho', 'fred', 'ben white', 'lucas paqueta']
def_names_set = set(df_def['join_name'].unique())

for t in targets:
    # We check if the TARGET name exists in our updated Main Join Keys
    # AND if it exists in the Defensive Keys
    in_main = t in df_main['join_name'].values
    in_def = t in def_names_set
    
    status = "✅ MATCHED" if (in_main and in_def) else "❌ STILL BROKEN"
    print(f"{t.ljust(15)} : {status}")

# Global Percentage Check
main_keys = set(zip(df_main['join_name'], df_main['GW'], df_main['join_season']))
def_keys = set(zip(df_def['join_name'], df_def['GW'], df_def['join_season']))
matched_rows = len(main_keys.intersection(def_keys))

print(f"\nNew Row Match Count: {matched_rows}")
print(f"Active Players (>0 Mins) still missing: {len(df_main[(~df_main['join_name'].isin(def_names_set)) & (df_main['minutes'] > 0)])}")

--- MAPPING MISSING STARS ---

=== FINAL STAR CHECK ===
joão moutinho   : ✅ MATCHED
fabinho         : ✅ MATCHED
fred            : ✅ MATCHED
ben white       : ✅ MATCHED
lucas paqueta   : ❌ STILL BROKEN

New Row Match Count: 40424
Active Players (>0 Mins) still missing: 1522


In [111]:
import unicodedata

# ==========================================
# 1. DEFINE ACCENT REMOVER
# ==========================================
def remove_accents(input_str):
    if not isinstance(input_str, str):
        return str(input_str)
    # Normalize unicode characters to decompose them (e.g., 'á' becomes 'a' + '´')
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    # Filter out non-spacing mark characters (the accents)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

print("--- STRIPPING ACCENTS FROM BOTH DATASETS ---")

# Apply to Main
df_main['join_name'] = df_main['join_name'].apply(remove_accents)

# Apply to Defensive
df_def['join_name'] = df_def['join_name'].apply(remove_accents)

# ==========================================
# 2. VERIFY PAQUETA & OTHERS
# ==========================================
print("\n--- ACCENT CHECK ---")
check_list = ['lucas paqueta', 'abdoulaye doucoure', 'lukasz fabianski']

for name in check_list:
    in_main = name in df_main['join_name'].values
    in_def = name in df_def['join_name'].values
    status = "✅ MATCHED" if (in_main and in_def) else "❌ MISMATCH"
    print(f"'{name}': {status}")

# ==========================================
# 3. UNIQUE NAME MATCHING STATS (The metric you care about)
# ==========================================
print("\n=== UNIQUE NAME COVERAGE REPORT ===")

# Get unique names
main_names_set = set(df_main['join_name'].unique())
def_names_set = set(df_def['join_name'].unique())

# Calculate intersection
matched_names = main_names_set.intersection(def_names_set)
missing_def_names = def_names_set - main_names_set

# Metrics
total_def_names = len(def_names_set)
matched_count = len(matched_names)
coverage_pct = (matched_count / total_def_names) * 100

print(f"Unique Defensive Names:   {total_def_names}")
print(f"Found in Main DataFrame:  {matched_count}")
print(f"Defensive Name Coverage:  {coverage_pct:.2f}%")

# Show the remaining names that STILL don't match
if len(missing_def_names) > 0:
    print("\n--- TOP MISSING NAMES (FROM DEFENSIVE FILE) ---")
    print(f"Total Missing: {len(missing_def_names)}")
    print("Sample of missing names (Are these important?):")
    print(sorted(list(missing_def_names))[:20])

--- STRIPPING ACCENTS FROM BOTH DATASETS ---

--- ACCENT CHECK ---
'lucas paqueta': ✅ MATCHED
'abdoulaye doucoure': ✅ MATCHED
'lukasz fabianski': ❌ MISMATCH

=== UNIQUE NAME COVERAGE REPORT ===
Unique Defensive Names:   1342
Found in Main DataFrame:  1271
Defensive Name Coverage:  94.71%

--- TOP MISSING NAMES (FROM DEFENSIVE FILE) ---
Total Missing: 71
Sample of missing names (Are these important?):
['abdukodir khusanov', 'abdul fatawu issahaku', 'ahmed hegazi', 'albert grønbaek', 'alex moreno', 'alex palmer', 'andres garcia', 'angelino', 'ansu fati', 'ben gannon-doak', 'beto', 'borja baston', 'cafu', 'chidozie obi-martin', 'chiquinho', 'claudio echeverri', 'cucho', 'dani ceballos', 'donyell malen', 'edmond-paris maghoma']


In [112]:

# ==========================================
# 1. FINAL MANUAL MAP (For Extreme Nicknames)
# ==========================================
# We map the Main (Legal) Name to the Defensive (Common) Name
# I found the legal names for the players in your missing list
final_nickname_map = {
    'norberto bercique gomes betuncal': 'beto',
    'jose angel esmoris tasende': 'angelino',
    'juan camilo hernandez suarez': 'cucho',
    'daniel ceballos fernandez': 'dani ceballos',
    'anssumane fati vieira': 'ansu fati',
    'anssumane fati': 'ansu fati',
    'alexandre moreno lopera': 'alex moreno',
    'łukasz fabianski': 'lukasz fabianski', # Fix the Polish 'ł' manually
    'lukasz fabianski': 'lukasz fabianski',  # Safety net
    'ahmed el-sayed hegazy': 'ahmed hegazi',
    'a\x81lex moreno lopera': 'alex moreno',   # Found in your list
    'ivan peria¡ia\x87': 'ivan perisic',       # Found in your list
    'muhamed bea¡ia\x87': 'muhamed besic',     # Found in your list
    'a\x81lex moreno lopera': 'alex moreno',
    'edson a\x81lvarez velazquez': 'edson alvarez',
    
    # The Nicknames & Legal Names
    'abdul fatawu': 'abdul fatawu issahaku',
    'borja gonzalez tomas': 'borja baston',
    'fabio ferreira vieira': 'fabio vieira',
    'fernando luiz rosa': 'fernandinho',
    'giovanni reyna': 'gio reyna',
    'hamed traore': 'hamed junior traore',
    'ian carlo poveda-ocampo': 'ian poveda',
    'jhon duran': 'jader duran',
    'julian araujo zuniga': 'julian araujo',
    'thakgalo leshabela': 'khanya leshabela',
    'francisco casilla cortes': 'kiko casilla',
    'francisco femenia far': 'kiko femenia',
    'marcus oliveira alencar': 'marquinhos',
    'oluwasemilogo adesewo ibidapo ajayi': 'semi ajayi',
    'tariqe fosu-henry': 'tariqe fosu',
    'vini de souza costa': 'vinicius souza',
    'vitor ferreira': 'vitinha',
    'jose reina': 'pepe reina',
    'djordje petrovic': 'đorđe petrovic',  # Matching the defensive spelling
    'jose a\x81ngel esmoris tasende': 'angelino', # Found hidden in candidate list

    
}

print("Applying final manual fixes...")
df_main['join_name'] = df_main['join_name'].replace(final_nickname_map)


Applying final manual fixes...


In [113]:
print("\n=== UNIQUE NAME COVERAGE REPORT ===")

# Get unique names
main_names_set = set(df_main['join_name'].unique())
def_names_set = set(df_def['join_name'].unique())

# Calculate intersection
matched_names = main_names_set.intersection(def_names_set)
missing_def_names = def_names_set - main_names_set

# Metrics
total_def_names = len(def_names_set)
matched_count = len(matched_names)
coverage_pct = (matched_count / total_def_names) * 100

print(f"Unique Defensive Names:   {total_def_names}")
print(f"Found in Main DataFrame:  {matched_count}")
print(f"Defensive Name Coverage:  {coverage_pct:.2f}%")


=== UNIQUE NAME COVERAGE REPORT ===
Unique Defensive Names:   1342
Found in Main DataFrame:  1299
Defensive Name Coverage:  96.80%


In [114]:
# ==========================================
# DIAGNOSTIC: FIND CANDIDATES FOR LEFTOVERS
# ==========================================

# 1. IDENTIFY WHO IS LEFT
# Defensive names that were NOT found in Main
main_names_set = set(df_main['join_name'].unique())
def_names_set = set(df_def['join_name'].unique())
missing_def_names = list(def_names_set - main_names_set)

# Main names that have NOT been matched to anyone yet
# We only care about ACTIVE players (Minutes > 0) to reduce noise
unused_main_df = df_main[~df_main['join_name'].isin(def_names_set)]
unused_main_names = unused_main_df[unused_main_df['minutes'] > 0]['join_name'].unique().tolist()

print(f"Searching for matches among:")
print(f" - {len(missing_def_names)} Unmatched Defensive Names")
print(f" - {len(unused_main_names)} Unmatched Active Main Names")
print("\n--- POTENTIAL CANDIDATES ---")
print(f"{'DEFENSIVE NAME (Short)':<30} | {'BEST CANDIDATES IN MAIN (Legal/Long)'}")
print("-" * 75)

# 2. FIND CANDIDATES
# For every missing defensive name, we look for "somewhat similar" names in the Main list
# We use a lower cutoff (0.4) just to see *any* potential connection
count = 0
for d_name in sorted(missing_def_names):
    # Get top 3 close matches from the unused main names
    matches = difflib.get_close_matches(d_name, unused_main_names, n=3, cutoff=0.4)
    
    if matches:
        # Join them with a comma for display
        candidates_str = ", ".join(matches)
        print(f"{d_name:<30} | {candidates_str}")
        count += 1
    else:
        # If no fuzzy match, maybe it's a completely different name (Beto/Norberto)
        # We print it alone so you know it has NO obvious match
        print(f"{d_name:<30} | (No similar spelling found)")

print("-" * 75)
print(f"Displayed {len(missing_def_names)} missing players.")

Searching for matches among:
 - 43 Unmatched Defensive Names
 - 25 Unmatched Active Main Names

--- POTENTIAL CANDIDATES ---
DEFENSIVE NAME (Short)         | BEST CANDIDATES IN MAIN (Legal/Long)
---------------------------------------------------------------------------
abdukodir khusanov             | (No similar spelling found)
albert grønbaek                | ben doak, carlos ribeiro dias
alex palmer                    | aorae petrovia
andres garcia                  | mateo kovaaia, lukasz fabianski, aorae petrovia
ben gannon-doak                | ben doak
cafu                           | (No similar spelling found)
chidozie obi-martin            | joe ayodele-aribo, mateus cardoso lemos martins
chiquinho                      | (No similar spelling found)
claudio echeverri              | carlos vinicius alves morais
donyell malen                  | (No similar spelling found)
edmond-paris maghoma           | paris maghoma, solomon march
eiran cashin                   | diogo

In [115]:
df_main.duplicated(
    subset=["join_name", "season", "GW"],
    keep=False
).sum()



np.int64(14155)

In [116]:
df_def.duplicated(
    subset=["join_name", "season", "GW"],
    keep=False
).sum()


np.int64(25428)

In [117]:
key_cols = ["join_name", "season", "GW"]

dup_mask = df_def.duplicated(subset=key_cols, keep=False)

print("Number of duplicated rows:", dup_mask.sum())
duplicated_rows = df_def[dup_mask].sort_values(by=key_cols)
print(duplicated_rows.head(10))

Number of duplicated rows: 25428
       season            name                    team position  minutes  \
5143  2019-20  Aaron Connolly  Brighton & Hove Albion       FW     67.0   
5503  2019-20  Aaron Connolly  Brighton & Hove Albion       FW     45.0   
6059  2019-20  Aaron Connolly  Brighton & Hove Albion       FW     67.0   
6328  2019-20  Aaron Connolly  Brighton & Hove Albion       FW     31.0   
8065  2019-20  Aaron Connolly  Brighton & Hove Albion       RW     20.0   
8259  2019-20  Aaron Connolly  Brighton & Hove Albion       FW     81.0   
8638  2019-20  Aaron Connolly  Brighton & Hove Albion       FW     86.0   
8943  2019-20  Aaron Connolly  Brighton & Hove Albion       FW     71.0   
9233  2019-20  Aaron Connolly  Brighton & Hove Albion       RM     20.0   
9433  2019-20  Aaron Connolly  Brighton & Hove Albion       FW     58.0   

      tackles  tackles_won  tackles_total  challenges  challenges_attempted  \
5143        0            0              0           0         

In [ ]:
# ==========================================
# 2. THE MERGE & SMART VALIDATION
# ==========================================
# IMPORTANT: We now use game_number instead of GW for merging
# This handles postponed matches correctly by matching on chronological game order
print("--- MERGE DIAGNOSTICS (Using game_number) ---")

# 1. PREPARE DEFENSIVE DATA
cols_cbi = ['clearances', 'blocks', 'interceptions']
df_def[cols_cbi] = df_def[cols_cbi].fillna(0)

if 'clearances_blocks_interceptions' not in df_def.columns:
    df_def['clearances_blocks_interceptions'] = (
        df_def['clearances'] + df_def['blocks'] + df_def['interceptions']
    )

# Select Merge Subset - NOW USING game_number INSTEAD OF GW
def_subset = df_def[[
    'join_name', 'game_number', 'join_season', 
    'tackles', 'clearances_blocks_interceptions'
]].rename(columns={
    'tackles': 'tackles_new', 
    'clearances_blocks_interceptions': 'cbi_new'
})

# ---------------------------------------------------------
# SMART METRIC: "Can we match it?"
# ---------------------------------------------------------
# Using game_number for matching ensures correct alignment even with postponed matches
main_keys = set(zip(df_main['join_name'], df_main['game_number'], df_main['join_season']))
def_keys = set(zip(def_subset['join_name'], def_subset['game_number'], def_subset['join_season']))

# The Intersection: These are the rows that SHOULD merge successfully
possible_matches = main_keys.intersection(def_keys)
print(f"Total Rows in Main: {len(df_main)}")
print(f"Rows with available Defensive Data: {len(possible_matches)}")

# 2. PERFORM LEFT MERGE - NOW ON game_number
merged_df = pd.merge(
    df_main, 
    def_subset, 
    on=['join_name', 'game_number', 'join_season'], 
    how='left'
)

# ---------------------------------------------------------
# REAL VALIDATION: DID THE MERGE WORK?
# ---------------------------------------------------------
merged_df['key_tuple'] = list(zip(merged_df['join_name'], merged_df['game_number'], merged_df['join_season']))
should_have_data = merged_df[merged_df['key_tuple'].isin(possible_matches)]

# Check if they are actually filled
successful_merges = should_have_data['tackles_new'].notna().sum()
technical_success_rate = (successful_merges / len(should_have_data)) * 100 if len(should_have_data) > 0 else 0

print(f"\nTechnical Merge Success Rate: {technical_success_rate:.2f}%")
print("(This should be 100%. It means every row that existed in the source was successfully merged.)")

# ==========================================
# 3. UPDATE STATS & FINAL REPORT
# ==========================================
# Update Tackles
merged_df['tackles'] = np.where(
    merged_df['tackles_new'].notna(), 
    merged_df['tackles_new'], 
    np.where(merged_df['minutes'] == 0, 0, merged_df['tackles'].fillna(0))
)

# Update CBI
old_cbi = merged_df['clearances_blocks_interceptions'] if 'clearances_blocks_interceptions' in merged_df.columns else 0
merged_df['clearances_blocks_interceptions'] = np.where(
    merged_df['cbi_new'].notna(), 
    merged_df['cbi_new'], 
    np.where(merged_df['minutes'] == 0, 0, old_cbi)
)

# Clean up temps - keep game_number as it's useful for feature engineering
merged_df.drop(columns=['tackles_new', 'cbi_new', 'join_name', 'join_season', 'is_matched', 'key_tuple'], inplace=True, errors='ignore')

print("\n✓ Merge complete using game_number for correct chronological alignment")

--- MERGE DIAGNOSTICS ---
Total Rows in Main: 142780
Rows with available Defensive Data: 40869

Technical Merge Success Rate: 100.00%
(This should be 100%. It means every row that existed in the source was successfully merged.)


In [119]:
print("==========================================")
print("       POST-MERGE DATA HEALTH CHECK       ")
print("==========================================")

# ---------------------------------------------------------
# 1. MACRO OVERVIEW
# ---------------------------------------------------------
target_cols = ['tackles', 'clearances_blocks_interceptions']

print(f"Total Rows: {len(merged_df):,}")
print("\n--- Missing Values (Global) ---")
print(merged_df[target_cols].isna().sum())

print("\n--- Zero Values (Global) ---")
# Count how many are exactly 0 (which might indicate successful fillna(0))
print((merged_df[target_cols] == 0).sum())

# ---------------------------------------------------------
# 2. THE "DANGER ZONE" ANALYSIS
# ---------------------------------------------------------
# We care most about rows where a player PLAYED (Minutes > 0) 
# but still has NaN in the stats. This indicates a data gap.
active_players = merged_df[merged_df['minutes'] > 0].copy()

print(f"\n--- Danger Zone: Active Players (Min > 0) with Missing Data ---")
print(f"Total Active Player-Matches: {len(active_players):,}")

for col in target_cols:
    missing_count = active_players[col].isna().sum()
    missing_pct = (missing_count / len(active_players)) * 100
    print(f" > {col}: {missing_count} missing ({missing_pct:.2f}%)")

# Optional: See WHICH seasons or teams are causing the missing data
if missing_count > 0:
    print(f"\n[Drill Down] Breakdown of missing {target_cols[0]} by Season:")
    missing_rows = active_players[active_players[target_cols[0]].isna()]
    print(missing_rows['season'].value_counts(dropna=False).head()) # Adjust 'season' to your actual season col name

# ---------------------------------------------------------
# 3. LOGIC VERIFICATION
# ---------------------------------------------------------
# You used logic to set stats to 0 if minutes == 0. Let's verify that stuck.
bench_players = merged_df[merged_df['minutes'] == 0]
non_zero_bench_stats = bench_players[target_cols].sum().sum()

print("\n--- Logic Check ---")
if non_zero_bench_stats == 0:
    print("✅ SUCCESS: All players with 0 minutes have 0 stats.")
else:
    print(f"⚠️ WARNING: Found {non_zero_bench_stats} non-zero stats for players with 0 minutes.")

# ---------------------------------------------------------
# 4. PLAYER SPOT CHECK
# ---------------------------------------------------------
def inspect_player(name_fragment, df=merged_df):
    """
    Filters for a player name and shows relevant columns to eyeball the merge.
    """
    # Case-insensitive search
    mask = df['name'].str.contains(name_fragment, case=False, na=False)
    subset = df[mask].sort_values(by=['season', 'GW']) # Adjust 'season' if needed
    
    cols_to_show = ['name', 'season', 'GW', 'minutes'] + target_cols
    
    # Handle cases where column names might vary slightly
    actual_cols = [c for c in cols_to_show if c in df.columns]
    
    print(f"\n--- Spot Check: {name_fragment} ---")
    if subset.empty:
        print("No player found.")
    else:
        # Show a sample of rows where they actually played
        print(subset[actual_cols].head(5).to_string(index=False))
        print(f"... ({len(subset)} total rows found)")

# Run spot checks on a Defender (high stats) and a Forward (low stats)
inspect_player("Trippier") # Good for checking tackles/CBI
inspect_player("Haaland")  # Should exist but have lower def stats

       POST-MERGE DATA HEALTH CHECK       
Total Rows: 154,894

--- Missing Values (Global) ---
tackles                            0
clearances_blocks_interceptions    0
dtype: int64

--- Zero Values (Global) ---
tackles                            123615
clearances_blocks_interceptions    113272
dtype: int64

--- Danger Zone: Active Players (Min > 0) with Missing Data ---
Total Active Player-Matches: 70,268
 > tackles: 0 missing (0.00%)
 > clearances_blocks_interceptions: 0 missing (0.00%)

--- Logic Check ---
⚠️ WARNING: Found 46609.0 non-zero stats for players with 0 minutes.

--- Spot Check: Trippier ---
               name  season  GW  minutes  tackles  clearances_blocks_interceptions
Kieran_Trippier_334 2019-20   1        0      0.0                              0.0
Kieran_Trippier_334 2019-20   2        0      0.0                              0.0
Kieran_Trippier_334 2019-20   3        0      0.0                              0.0
Kieran_Trippier_334 2019-20   4        0      0.0    

In [120]:
# 1. Create a mask for the "Impossible" rows
# (Minutes are 0, but they have stats > 0)
ghost_mask = (merged_df['minutes'] == 0) & (
    (merged_df['tackles'] > 0) | 
    (merged_df['clearances_blocks_interceptions'] > 0)
)

# 2. Grab the subset
ghost_rows = merged_df[ghost_mask]

# 3. specific columns to help diagnose
cols_to_check = [
    'name', 'team', 'season', 'GW', 
    'minutes', 'tackles', 'clearances_blocks_interceptions'
]
# Adjust 'season' or 'team' if your column names differ slightly

print(f"Total Ghost Rows: {len(ghost_rows)}")
print("\n--- Sample of Ghost Rows ---")
print(ghost_rows[cols_to_check].sample(15).to_string(index=False))

# 4. Optional: Check if specific seasons are worse
print("\n--- Breakdown by Season ---")
print(ghost_rows['season'].value_counts())

Total Ghost Rows: 10078

--- Sample of Ghost Rows ---
                               name        team  season  GW  minutes  tackles  clearances_blocks_interceptions
                    NicolÃ² Zaniolo Aston Villa 2023-24  36        0      1.0                              0.0
                    GaÃ«tan_Bong_41    Brighton 2019-20   9        0      1.0                              1.0
                   Jamaal Lascelles   Newcastle 2021-22   7        0      0.0                             10.0
                    Jamie_Vardy_166   Leicester 2019-20  20        0      1.0                              2.0
                     Tim Iroegbunam Aston Villa 2021-22  38        0      1.0                              0.0
Cristiano Ronaldo dos Santos Aveiro     Man Utd 2021-22  22        0      0.0                              1.0
                     Mohammed Kudus    West Ham 2024-25  12        0      2.0                              0.0
                        Luke Thomas   Leicester 2021-22   

In [ ]:
# ==========================================
# SAVE MERGED DATA WITH GAME_NUMBER
# ==========================================
print("--- SAVING FINAL MERGED DATA ---")

# Verify game_number column exists
if 'game_number' in merged_df.columns:
    print(f"✓ game_number column present. Range: {merged_df['game_number'].min()} to {merged_df['game_number'].max()}")
else:
    print("⚠️ game_number column not found! Creating it now...")
    merged_df = merged_df.sort_values(['name', 'season', 'GW', 'fixture']).reset_index(drop=True)
    merged_df['game_number'] = merged_df.groupby(['name', 'season']).cumcount() + 1

# Save to CSV
output_file = 'all_seasons_data_with_defensive.csv'
merged_df.to_csv(output_file, index=False)
print(f"✓ Saved: {output_file}")
print(f"  Shape: {merged_df.shape}")
print(f"  Columns include: game_number, GW, tackles, clearances_blocks_interceptions")

# Quick summary
print(f"\n=== FINAL DATA SUMMARY ===")
print(f"Total rows: {len(merged_df):,}")
print(f"Unique players: {merged_df['name'].nunique():,}")
print(f"Seasons: {sorted(merged_df['season'].unique())}")
print(f"game_number range per season: 1 to {merged_df.groupby('season')['game_number'].max().max()}")